# 19. Does the Specific Evidence Geometry Feature Design Matter?

`DESIGN.md` §17's Evidence Geometry features (`msp`, `logit_margin`, `normalized_entropy`, `energy_score`,
`logit_l2_norm` — `src/deployment_reliability/features.py`) were hand-designed, each targeting a
specific, named failure mode (§6). An external critique raised the obvious control this project had not
yet run: is the *design* doing any real work, or would **any** 5-dimensional summary of the logit vector
combined with the same learned `LogisticRegressionCombiner` do about as well?

This notebook tests that directly, on ResNet-50's cached data (`data/logit_cache_resnet50.pt`), with two
generic 5-dimensional alternatives to `featurize(logits)`:

1. **PCA-5** — the top 5 principal components of the raw 1000-way logit vector.
2. **Random projection-5** — 5 fixed random linear projections of the raw logit vector (Johnson-
   Lindenstrauss-style, no data-dependent fitting beyond the random matrix itself).

**Design decision, stated up front:** PCA and the random projections are both computed on the raw
**logit vectors directly**, not on `phi(z)`'s 5 hand-designed features. Fitting PCA on `phi(z)` would
only ever recover a rotation/rescaling of the same 5 already-designed quantities — it couldn't test the
actual question, which is whether the *hand-designed, nonlinear* feature choices (softmax max, top-2
gap, normalized entropy, log-sum-exp, L2 norm) add anything over a *generic linear* summary of the same
1000-dimensional raw signal the features were built from.

All fitting (PCA's mean/directions, the random projection matrix's — trivial, since it's fixed by seed
and never touches labels —, and `LogisticRegressionCombiner`'s weights) uses **only** `combiner_fit`,
identically to how `featurize`'s real features are fit elsewhere in this project (`DESIGN.md` §10.5);
`id_test` is evaluated on only, never fit on, for all three representations.

In [1]:
import os
import sys

import numpy as np
import torch

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
torch.manual_seed(0)

from deployment_reliability.combiner import LogisticRegressionCombiner
from deployment_reliability.features import DEFAULT_FEATURE_NAMES, featurize
from deployment_reliability.router import aurc, auroc
from deployment_reliability.significance import bootstrap_auroc_ci

CACHE_PATH = os.path.join("..", "data", "logit_cache_resnet50.pt")
assert os.path.exists(CACHE_PATH), "run scripts/collect_logits.py resnet50 first"
cache = torch.load(CACHE_PATH)
logits, labels = cache["logits"], cache["labels"]
splits_arr = np.array(cache["splits"])

def mask(name):
    return torch.from_numpy(splits_arr == name)

m_fit, m_cal, m_test, m_a, m_o = [mask(n) for n in ("combiner_fit", "threshold_cal", "id_test", "imagenet_a", "imagenet_o")]
predicted = logits.argmax(dim=-1)
correct = predicted == labels
print(f"combiner_fit n={int(m_fit.sum())}  id_test n={int(m_test.sum())}  num_classes={logits.shape[-1]}")


combiner_fit n=1500  id_test n=1500  num_classes=1000


## Build the three 5-dimensional representations

In [2]:
# 1. Real Evidence Geometry features (DESIGN.md sections 5-6)
phi_real = featurize(logits)

# 2. PCA-5: mean and top-5 principal directions fit on combiner_fit's RAW logits only
#    (centering only, i.e. covariance-based PCA - no per-dimension rescaling, since all
#    1000 logit dimensions share the same underlying unit).
logit_mean = logits[m_fit].mean(dim=0, keepdim=True)
centered_fit = logits[m_fit] - logit_mean
_, singular_values, Vt = torch.linalg.svd(centered_fit, full_matrices=False)
pca_components = Vt[:5]  # (5, num_classes) - top-5 principal directions
phi_pca = (logits - logit_mean) @ pca_components.T  # (N, 5)

explained_variance_ratio = (singular_values[:5] ** 2 / (singular_values ** 2).sum()).tolist()
print("PCA-5 explained variance ratio (top 5 of", logits.shape[-1], "components):",
      [f"{v:.4f}" for v in explained_variance_ratio], " sum =", f"{sum(explained_variance_ratio):.4f}")

# 3. Random projection-5: FIXED seed=42, generated once, never refit or touched by labels.
#    Scaled by 1/sqrt(num_classes) (Johnson-Lindenstrauss-style) so its output magnitude is
#    comparable to a typical logit-space linear functional, not a free calibration choice.
rand_generator = torch.Generator().manual_seed(42)
random_projection = torch.randn(logits.shape[-1], 5, generator=rand_generator) / (logits.shape[-1] ** 0.5)
phi_random = logits @ random_projection  # (N, 5)

print(f"\nphi_real shape={tuple(phi_real.shape)}  phi_pca shape={tuple(phi_pca.shape)}  phi_random shape={tuple(phi_random.shape)}")


PCA-5 explained variance ratio (top 5 of

 1000 components): ['0.0860', '0.0797', '0.0645', '0.0536', '0.0464']  sum = 0.3302

phi_real shape=(7235, 5)  phi_pca shape=(7235, 5)  phi_random shape=(7235, 5)


## Fit `LogisticRegressionCombiner` identically on each, evaluate on `id_test`

Same protocol for all three: fit on `combiner_fit`, evaluate on `id_test` (correctness AUROC/AURC) and
`id_test` vs. `imagenet_o` (OOD AUROC, as a secondary check) - never fit on anything but `combiner_fit`.

In [3]:
def fit_and_evaluate(phi, label):
    combiner = LogisticRegressionCombiner().fit(phi[m_fit], correct[m_fit].float())
    s = combiner.score(phi)
    s_test = s[m_test]
    correct_test = correct[m_test]
    ci = bootstrap_auroc_ci(s_test[correct_test], s_test[~correct_test], n_bootstrap=3000, seed=0)
    result = {
        "label": label,
        "aurc_test": aurc(s_test, correct_test),
        "auroc_corr_id": ci.auroc,
        "auroc_ci_lo": ci.ci_lo,
        "auroc_ci_hi": ci.ci_hi,
        "auroc_ood_o": auroc(s_test, s[m_o]),
    }
    return result

results = [
    fit_and_evaluate(phi_real, "Evidence Geometry (5 hand-designed features)"),
    fit_and_evaluate(phi_pca, "PCA-5 of raw logits"),
    fit_and_evaluate(phi_random, "Random projection-5 of raw logits"),
]

print(f"{'representation':46s} {'AURC(id_test)':>14s} {'AUROC(corr_id)':>15s} {'95% CI':>18s} {'AUROC(id vs O)':>15s}")
for r in results:
    ci_str = f"[{r['auroc_ci_lo']:.4f}, {r['auroc_ci_hi']:.4f}]"
    print(f"{r['label']:46s} {r['aurc_test']:14.4f} {r['auroc_corr_id']:15.4f} {ci_str:>18s} {r['auroc_ood_o']:15.4f}")


representation                                  AURC(id_test)  AUROC(corr_id)             95% CI  AUROC(id vs O)
Evidence Geometry (5 hand-designed features)           0.0430          0.8841   [0.8664, 0.9005]          0.5673
PCA-5 of raw logits                                    0.1041          0.7791   [0.7447, 0.8119]          0.4458
Random projection-5 of raw logits                      0.1233          0.6812   [0.6458, 0.7163]          0.3265


## Robustness check: is the random-projection result sensitive to the seed?

A single random draw could get unusually lucky or unlucky. Repeat the random-projection fit/eval across
10 independent seeds and report the mean +/- std, to confirm the comparison above isn't an artifact of
`seed=42` specifically.

In [4]:
random_seed_aurocs = []
random_seed_aurcs = []
for seed in range(10):
    g = torch.Generator().manual_seed(seed)
    proj = torch.randn(logits.shape[-1], 5, generator=g) / (logits.shape[-1] ** 0.5)
    phi_r = logits @ proj
    r = fit_and_evaluate(phi_r, f"random seed={seed}")
    random_seed_aurocs.append(r["auroc_corr_id"])
    random_seed_aurcs.append(r["aurc_test"])
    print(f"  seed={seed}  AUROC(corr_id)={r['auroc_corr_id']:.4f}  AURC(id_test)={r['aurc_test']:.4f}")

random_seed_aurocs = np.array(random_seed_aurocs)
random_seed_aurcs = np.array(random_seed_aurcs)
print(f"\nacross 10 seeds: AUROC(corr_id) = {random_seed_aurocs.mean():.4f} +/- {random_seed_aurocs.std():.4f}"
      f"  (range [{random_seed_aurocs.min():.4f}, {random_seed_aurocs.max():.4f}])")
print(f"across 10 seeds: AURC(id_test)  = {random_seed_aurcs.mean():.4f} +/- {random_seed_aurcs.std():.4f}"
      f"  (range [{random_seed_aurcs.min():.4f}, {random_seed_aurcs.max():.4f}])")

real_auroc = results[0]["auroc_corr_id"]
gap_in_stds = (real_auroc - random_seed_aurocs.mean()) / random_seed_aurocs.std()
print(f"\nEvidence Geometry's AUROC ({real_auroc:.4f}) is {gap_in_stds:.1f} random-projection standard deviations above the random-projection mean.")


  seed=0  AUROC(corr_id)=0.6864  AURC(id_test)=0.1226


  seed=1  AUROC(corr_id)=0.7399  AURC(id_test)=0.1102


  seed=2  AUROC(corr_id)=0.6667  AURC(id_test)=0.1305


  seed=3  AUROC(corr_id)=0.6879  AURC(id_test)=0.1212


  seed=4  AUROC(corr_id)=0.6844  AURC(id_test)=0.1282


  seed=5  AUROC(corr_id)=0.6466  AURC(id_test)=0.1463


  seed=6  AUROC(corr_id)=0.6831  AURC(id_test)=0.1234


  seed=7  AUROC(corr_id)=0.7056  AURC(id_test)=0.1068


  seed=8  AUROC(corr_id)=0.6329  AURC(id_test)=0.1615


  seed=9  AUROC(corr_id)=0.7520  AURC(id_test)=0.0860

across 10 seeds: AUROC(corr_id) = 0.6886 +/- 0.0351  (range [0.6329, 0.7520])
across 10 seeds: AURC(id_test)  = 0.1237 +/- 0.0198  (range [0.0860, 0.1615])

Evidence Geometry's AUROC (0.8841) is 5.6 random-projection standard deviations above the random-projection mean.


## Result, stated plainly

The comparison above is the real, computed answer - reported as found, whichever way it went.

In [5]:
real_r, pca_r, rand_r = results
print("=== Head-to-head, ResNet-50, id_test ===")
print(f"Evidence Geometry (5 features): AURC={real_r['aurc_test']:.4f}  AUROC(corr_id)={real_r['auroc_corr_id']:.4f}")
print(f"PCA-5 of raw logits:            AURC={pca_r['aurc_test']:.4f}  AUROC(corr_id)={pca_r['auroc_corr_id']:.4f}")
print(f"Random projection-5:            AURC={rand_r['aurc_test']:.4f}  AUROC(corr_id)={rand_r['auroc_corr_id']:.4f} (mean of 10 seeds: {random_seed_aurocs.mean():.4f})")

auroc_gap_pca = real_r["auroc_corr_id"] - pca_r["auroc_corr_id"]
auroc_gap_rand = real_r["auroc_corr_id"] - random_seed_aurocs.mean()
print(f"\nEvidence Geometry beats PCA-5 by {auroc_gap_pca:+.4f} AUROC, and beats random projection-5 by {auroc_gap_rand:+.4f} AUROC (mean over 10 seeds).")

if auroc_gap_pca > 0.02 and auroc_gap_rand > 0.02:
    print("\nCONCLUSION: the hand-designed Evidence Geometry feature set clearly outperforms both generic")
    print("dimensionality-reduction baselines at the same dimensionality (5) on the same data. The specific")
    print("nonlinear constructions (softmax max, top-2 logit gap, normalized entropy, log-sum-exp energy,")
    print("L2 norm) are carrying real, checkable value beyond 'any 5-dimensional summary of the logits' -")
    print("this is a genuine validation of the feature design, not an assumption.")
elif abs(auroc_gap_pca) < 0.01 and abs(auroc_gap_rand) < 0.01:
    print("\nCONCLUSION: PCA-5 and/or random projection-5 come within noise of the hand-designed feature")
    print("set's performance. This is a genuine, disclosable finding against the specific feature design's")
    print("necessity - a generic 5-dimensional linear summary of the logits does nearly as well here, and")
    print("the marginal value of the hand-crafted nonlinear constructions specifically should be reported")
    print("as smaller than DESIGN.md's feature-design narrative has assumed.")
else:
    print("\nCONCLUSION: a real but partial edge for the hand-designed features - clearly ahead of one")
    print("generic baseline but not both by the same margin; reported with the exact numbers above rather")
    print("than rounded to a single verdict.")


=== Head-to-head, ResNet-50, id_test ===
Evidence Geometry (5 features): AURC=0.0430  AUROC(corr_id)=0.8841
PCA-5 of raw logits:            AURC=0.1041  AUROC(corr_id)=0.7791
Random projection-5:            AURC=0.1233  AUROC(corr_id)=0.6812 (mean of 10 seeds: 0.6886)

Evidence Geometry beats PCA-5 by +0.1050 AUROC, and beats random projection-5 by +0.1955 AUROC (mean over 10 seeds).

CONCLUSION: the hand-designed Evidence Geometry feature set clearly outperforms both generic
dimensionality-reduction baselines at the same dimensionality (5) on the same data. The specific
nonlinear constructions (softmax max, top-2 logit gap, normalized entropy, log-sum-exp energy,
L2 norm) are carrying real, checkable value beyond 'any 5-dimensional summary of the logits' -
this is a genuine validation of the feature design, not an assumption.
